# Formula 1 Prediction EDA, 2019-2025 Dataset

Exploratory analysis for `dataset/outputs/prediction_2014.csv` filtered to seasons 2019-2025.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.05)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

OUTPUT_PREFIX = 'eda2019'
TARGET_ORDER = ['podium', 'points', 'no_points']
TARGET_COLORS = ['#FFD700', '#C0C0C0', '#CD7F32']


def load_and_clean_data(filepath):
    print(f"Loading dataset from: {filepath}")
    df = pd.read_csv(filepath)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)
    print(f"Dataset loaded successfully. Shape: {df.shape[0]} rows, {df.shape[1]} columns.")
    return df


def savefig(name, **kwargs):
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_PREFIX}_{name}.png', bbox_inches='tight', **kwargs)
    plt.close()


def plot_missing_values(df):
    print('Generating Missing Values plot...')
    missing_pct = df.isnull().mean().mul(100)
    missing_cols = missing_pct[missing_pct > 0].sort_values(ascending=False)
    if missing_cols.empty:
        print('-> No NaN missing values detected. Sentinel -1 values are analyzed separately.')
        return
    plt.figure(figsize=(12, 6))
    sns.barplot(x=missing_cols.values, y=missing_cols.index, palette='Reds_r')
    plt.title('Percentage of NaN Missing Values per Feature', fontsize=16, fontweight='bold')
    plt.xlabel('% Missing')
    savefig('00_missing_values')


def plot_target_distribution(df):
    print('Generating Target Distribution...')
    plt.figure(figsize=(8, 6))
    ax = sns.countplot(data=df, x='target', hue='target', order=TARGET_ORDER, hue_order=TARGET_ORDER, palette=TARGET_COLORS, edgecolor='black', legend=False)
    plt.title('Target Variable Distribution, 2019-2025', fontsize=16, fontweight='bold')
    plt.xlabel('Race Outcome')
    plt.ylabel('Count')
    total = len(df)
    for p in ax.patches:
        count = int(p.get_height())
        ax.annotate(f'{count}\n{count / total:.1%}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontweight='bold', xytext=(0, 5), textcoords='offset points')
    savefig('01_target_distribution')


def plot_target_by_year(df):
    print('Generating Target Distribution by Year...')
    yearly = (
        pd.crosstab(df['year'], df['target'], normalize='index')
        .reindex(columns=TARGET_ORDER, fill_value=0)
        .reset_index()
        .melt(id_vars='year', var_name='target', value_name='rate')
    )
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=yearly, x='year', y='rate', hue='target', hue_order=TARGET_ORDER,
                 marker='o', linewidth=2.2, palette=TARGET_COLORS)
    plt.title('Target Class Rate by Season', fontsize=16, fontweight='bold')
    plt.xlabel('Season')
    plt.ylabel('Class rate within season')
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    plt.xticks(sorted(df['year'].unique()), rotation=45)
    savefig('02_target_by_year')


def plot_feature_distributions(df):
    print('Generating Continuous Feature Distributions...')
    features = [
        'driver_age', 'days_since_last_race', 'points_gap_to_leader',
        'driver_avg_position_last3', 'teammate_h2h_avg_position_delta', 'driver_dnf_rate_historical'
    ]
    axis_cfg = {
        'driver_age': {'xlim': (18, 45), 'xticks': np.arange(18, 46, 2), 'xlabel': 'Driver age (years)'},
        'days_since_last_race': {'xlim': (-2, 160), 'xticks': [-1, 0, 7, 14, 21, 28, 56, 84, 112, 140], 'xlabel': 'Days since previous race (-1 = no history)'},
        'points_gap_to_leader': {'xlim': (-5, 600), 'xticks': np.arange(0, 601, 50), 'xlabel': 'Points behind championship leader'},
        'driver_avg_position_last3': {'xlim': (-1, 22), 'xticks': [-1] + list(np.arange(1, 23, 1)), 'xlabel': 'Average finishing position, previous 3 races'},
        'teammate_h2h_avg_position_delta': {'xlim': (-18, 18), 'xticks': np.arange(-18, 19, 3), 'xlabel': 'Average position delta vs teammate'},
        'driver_dnf_rate_historical': {'xlim': (-1, 1), 'xticks': [-1] + list(np.round(np.arange(0, 1.01, 0.1), 1)), 'xlabel': 'Historical DNF rate (-1 = no history)'},
    }
    valid_features = [f for f in features if f in df.columns]
    fig, axes = plt.subplots(2, 3, figsize=(24, 14))
    fig.suptitle('Distributions of Key Predictors, 2019-2025', fontsize=20, fontweight='bold', y=1.02)
    axes = axes.flatten()
    for i, col in enumerate(valid_features):
        ax = axes[i]
        values = df[col].dropna()
        cfg = axis_cfg.get(col, {})
        xlim = cfg.get('xlim')
        plot_values = values
        clipped_count = 0
        if xlim is not None:
            clipped_count = int(((values < xlim[0]) | (values > xlim[1])).sum())
            plot_values = values[(values >= xlim[0]) & (values <= xlim[1])]
        sns.histplot(plot_values, kde=True, ax=ax, color='teal', bins=60, edgecolor='black', linewidth=0.4)
        median_value = values.median()
        ax.axvline(median_value, color='crimson', linestyle='--', linewidth=1.5, label=f'Median: {median_value:.2f}')
        ax.set_title(col, fontsize=13, fontweight='bold')
        ax.set_xlabel(cfg.get('xlabel', col), fontsize=10)
        ax.set_ylabel('Frequency')
        ax.grid(True, alpha=0.35)
        ax.minorticks_on()
        if xlim is not None:
            ax.set_xlim(*xlim)
        if 'xticks' in cfg:
            ax.set_xticks(cfg['xticks'])
            ax.tick_params(axis='x', labelrotation=45)
        stats_text = f"min={values.min():.2f}\nmax={values.max():.2f}"
        if clipped_count > 0:
            stats_text += f"\n{clipped_count} outside shown range"
        ax.text(0.98, 0.95, stats_text, transform=ax.transAxes, ha='right', va='top',
                fontsize=9, bbox=dict(facecolor='white', edgecolor='gray', alpha=0.85))
        ax.legend(loc='upper left', fontsize=9, frameon=True)
    for j in range(len(valid_features), len(axes)):
        fig.delaxes(axes[j])
    savefig('03_feature_distributions')


def plot_feature_separability(df):
    print('Generating Feature Separability Boxplots...')
    features_to_check = [
        'driver_avg_position_last3', 'driver_avg_position_last5',
        'driver_avg_position_last10', 'teammate_h2h_avg_position_delta'
    ]
    for feature in features_to_check:
        if feature not in df.columns:
            continue
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x='target', y=feature, hue='target', order=TARGET_ORDER, hue_order=TARGET_ORDER, palette='Set2', legend=False)
        plt.title(f'Target Separation by {feature}, 2019-2025', fontsize=16, fontweight='bold')
        plt.xlabel('Race Outcome')
        plt.ylabel(feature)
        savefig(f'04_separability_{feature}')


def plot_sentinel_minus_one_rates(df):
    print('Generating Sentinel -1 Rate plot...')
    numeric_df = df.select_dtypes(include=[np.number])
    ignore_cols = {'raceId', 'driverId', 'constructorId', 'circuitId', 'year', 'round'}
    rates = ((numeric_df.drop(columns=[c for c in ignore_cols if c in numeric_df.columns], errors='ignore') == -1).mean() * 100)
    rates = rates[rates > 0].sort_values(ascending=True)
    if rates.empty:
        print('-> No -1 sentinel values detected.')
        return
    plt.figure(figsize=(11, max(5, len(rates) * 0.35)))
    sns.barplot(x=rates.values, y=rates.index, palette='viridis')
    plt.title('Sentinel -1 Rate by Numeric Feature', fontsize=16, fontweight='bold')
    plt.xlabel('% rows equal to -1')
    plt.ylabel('Feature')
    for i, value in enumerate(rates.values):
        plt.text(value + 0.1, i, f'{value:.1f}%', va='center', fontsize=9)
    savefig('05_sentinel_minus_one_rates')


def plot_correlation_heatmap(df):
    print('Generating Spearman Correlation Heatmap...')
    numeric_df = df.select_dtypes(include=[np.number])
    ignore_cols = ['raceId', 'driverId', 'constructorId', 'circuitId', 'year', 'round']
    numeric_df = numeric_df.drop(columns=[c for c in ignore_cols if c in numeric_df.columns], errors='ignore')
    corr_matrix = numeric_df.corr(method='spearman')
    plt.figure(figsize=(20, 16))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0, vmax=1, vmin=-1,
                annot=False, square=True, linewidths=.5, cbar_kws={'shrink': .7})
    plt.title('Spearman Rank Correlation Heatmap, 2019-2025', fontsize=20, fontweight='bold')
    savefig('06_correlation_heatmap')


def plot_rf_feature_importance(df):
    print('Generating Baseline Random Forest Feature Importance...')
    drop_cols = ['date', 'year', 'round', 'target', 'raceId', 'driverId', 'constructorId', 'circuitId']
    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    y = df['target']
    X = X.fillna(X.median(numeric_only=True))
    rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
    rf.fit(X, y)
    importances = rf.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    top_n = min(25, len(X.columns))
    plt.figure(figsize=(12, 10))
    sns.barplot(x=importances[sorted_idx[:top_n]], y=X.columns[sorted_idx[:top_n]], palette='magma')
    plt.title(f'Top {top_n} Predictors, Baseline Random Forest', fontsize=16, fontweight='bold')
    plt.xlabel('Gini Importance')
    plt.ylabel('Feature')
    savefig('07_feature_importance')
    return pd.Series(importances, index=X.columns).sort_values(ascending=False)


In [ ]:
csv_path = '../dataset/outputs/prediction_2014.csv'
f1_df = load_and_clean_data(csv_path)
f1_df.head()


In [ ]:
plot_missing_values(f1_df)
plot_target_distribution(f1_df)
plot_target_by_year(f1_df)
plot_feature_distributions(f1_df)
plot_feature_separability(f1_df)
plot_sentinel_minus_one_rates(f1_df)
plot_correlation_heatmap(f1_df)
rf_importances = plot_rf_feature_importance(f1_df)
rf_importances.head(15)
